# Demand Forecasting from CSV (LightGBM + Naive + W-37)

In [ ]:
import os
from ibm_watsonx_ai import APIClient, Credentials
import getpass

credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key= "xxxxxx-xxxxxxxxxxxxxx" # use this API or create your API 
)


In [ ]:
client = APIClient(credentials)
space_id = "xxxxx-xxxx-xxx-xxxx-xxxxxxxxxx"  # # get you space_id
client.set.default_space(space_id)
source_project_id = project.get_metadata()['metadata']['guid']

In [ ]:

# Install required packages (run once, then restart kernel)
%pip install "numpy<2" pandas matplotlib lightgbm scikit-learn pyarrow


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lightgbm import LGBMRegressor, early_stopping


In [ ]:

import os, types
import pandas as pd
from botocore.client import Config
import ibm_boto3

def __iter__(self): return 0

# @hidden_cell
# The following code accesses a file in your IBM Cloud Object Storage. It includes your credentials.
# Replace the placeholders below with your own project's COS credentials, or use
    # Insert code -> Read data in the notebook UI to regenerate this cell for your environment.

cos_client = ibm_boto3.client(service_name='s3',
    ibm_api_key_id='PASTE_YOUR_COS_API_KEY',
    ibm_auth_endpoint="https://iam.cloud.ibm.com/identity/token",
    config=Config(signature_version='oauth'),
    endpoint_url='https://s3.direct.us-south.cloud-object-storage.appdomain.cloud')

bucket = 'PASTE_YOUR_BUCKET_NAME'
object_key = 'training_data_v2.csv'  # your uploaded training file

body = cos_client.get_object(Bucket=bucket,Key=object_key)['Body']
# add missing __iter__ method, so pandas accepts body as file-like object
if not hasattr(body, "__iter__"): body.__iter__ = types.MethodType( __iter__, body )

df_1 = pd.read_csv(body)
df_1.head(10)


In [ ]:
df_1 = df_1.iloc[50000:]


df_1["TXN_DATE"] = pd.to_datetime(df_1["TXN_DATE"], errors="coerce").dt.strftime("%Y-%m-%d")
cols_to_clean = df_1.columns.difference(["TXN_DATE"])

df_1[cols_to_clean] = (
    df_1[cols_to_clean]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0.0)
    .astype(float)
)



In [ ]:
df_1.info()


In [ ]:
df=df_1
df

In [ ]:

# Load CSV (CHANGE PATH)
#df = pd.read_csv("your_file.csv")

df["TXN_DATE"] = pd.to_datetime(df["TXN_DATE"], errors="coerce")
df = df.dropna(subset=["TXN_DATE"])
df = df.sort_values(["SEGMENT_ID", "TXN_DATE"]).reset_index(drop=True)

print(df.shape)
df.head()


In [ ]:

# Train / Validation split
X = df.drop(columns=["target", "TXN_DATE"])
y = df["target"]

split_year = X["year"].quantile(0.8)
mask = X["year"] <= split_year

X_train, X_val = X[mask], X[~mask]
y_train, y_val = y[mask], y[~mask]


In [ ]:
X_test = X.tail(1000)
y_test = y.tail(1000)


In [ ]:
X_test


In [ ]:
train_df = pd.concat([X_train, y_train.rename("target")], axis=1)
train_df.to_csv("training_data.csv", index=False)

test_df = pd.concat([X_test, y_test.rename("target")], axis=1)
test_df.to_csv("test_data.csv", index=False)


In [ ]:
project.save_data(
    data=open("/home/wsuser/work/test_data.csv", "rb"),
    file_name="test_data.csv",
    overwrite=True
)

In [ ]:
project.save_data(
    data=open("/home/wsuser/work/training_data.csv", "rb"),
    file_name="training_data.csv",
    overwrite=True
)

In [ ]:
from lightgbm import LGBMRegressor, early_stopping

# Train LightGBM (4.x compatible)
def train_lgbm_model(x_train, y_train, x_valid, y_valid):
    model = LGBMRegressor(
        objective="regression",
        n_estimators=1000,
        learning_rate=0.05,
        random_state=42,
        metric="rmse"
    )

    model.fit(
        x_train,
        y_train,
        eval_set=[(x_valid, y_valid)],
        eval_metric="rmse",
        callbacks=[early_stopping(50)],
    )
    return model

m_lgb = train_lgbm_model(X_train, y_train, X_val, y_val)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    explained_variance_score
)
from scipy.stats import pearsonr, spearmanr


def full_regression_report(
    y_true,
    y_pred,
    y_train_pred=None,
    business_tolerance_pct=0.15
):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)

    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    smape = (
        np.mean(
            2 * np.abs(y_pred - y_true)
            / (np.abs(y_true) + np.abs(y_pred))
        )
        * 100
    )

    r2 = r2_score(y_true, y_pred)
    explained_var = explained_variance_score(y_true, y_pred)

    pearson_corr, _ = pearsonr(y_true, y_pred)
    spearman_corr, _ = spearmanr(y_true, y_pred)

    mse_train = None
    if y_train_pred is not None:
        mse_train = mean_squared_error(y_true[:len(y_train_pred)], y_train_pred)
        
    def rate_pct(pct):
        if pct < 5:
            return "Excellent"
        elif pct < 10:
            return "Good"
        elif pct < business_tolerance_pct * 100:
            return "Acceptable"
        else:
            return "Poor"

    verdict = "✅ Production-ready"
    if mape > business_tolerance_pct * 100:
        verdict = "⚠️ Acceptable with monitoring"
    if r2 < 0.4:
        verdict = "❌ Not recommended"

    report = pd.DataFrame({
        "Metric": [
            "Spearman",
            "Proportion explained variance",
            "Pearson",
            "Symmetric mean absolute percentage error",
            "Root of mean squared error",
            "Mean absolute error",
            "Mean squared error (val)",
            "Mean squared error (train)",
            "Mean absolute percentage error",
            "R squared"
        ],
        "Value": [
            round(spearman_corr, 2),
            round(explained_var, 2),
            round(pearson_corr, 2),
            round(smape, 2),
            round(rmse, 2),
            round(mae, 2),
            round(mse, 2),
            round(mse_train, 2) if mse_train else None,
            round(mape, 2),
            round(r2, 2)
        ],
        "Recommendation": [
            "Strong monotonic relationship" if spearman_corr > 0.6 else "Weak",
            "Good variance capture" if explained_var > 0.4 else "Low",
            "Strong linear relationship" if pearson_corr > 0.7 else "Moderate",
            rate_pct(smape),
            rate_pct((rmse / np.mean(np.abs(y_true))) * 100),
            rate_pct((mae / np.mean(np.abs(y_true))) * 100),
            "Compare train vs val",
            "Baseline only" if mse_train else "N/A",
            rate_pct(mape),
            "Good" if r2 >= 0.5 else "Acceptable"
        ]
    })

    return report, verdict


In [ ]:
# Validation predictions
y_val_pred = m_lgb.predict(X_val)

# Generate report
metrics_table, final_verdict = full_regression_report(
    y_true=y_val,
    y_pred=y_val_pred
)

metrics_table


In [ ]:
from lightgbm import LGBMRegressor

# Predictions & actuals
prediction_df = X_val.copy()
prediction_df["TXN_DATE"] = df.loc[X_val.index, "TXN_DATE"]
prediction_df["prediction"] = m_lgb.predict(X_val)
prediction_df = prediction_df[["SEGMENT_ID", "TXN_DATE", "prediction"]]

actual_df = df.loc[X_val.index, ["SEGMENT_ID", "TXN_DATE"]].copy()
actual_df["actual"] = y_val.values


In [ ]:

# Naive (last month)
def same_as_last_month(df):
    d = df.sort_values(["SEGMENT_ID", "TXN_DATE"]).copy()
    d["naive_prediction"] = d.groupby("SEGMENT_ID")["TXN_VOLUME"].shift(1)
    return d[["SEGMENT_ID", "TXN_DATE", "naive_prediction"]].set_index(["SEGMENT_ID", "TXN_DATE"])

naive_df = same_as_last_month(df)


In [ ]:

# W-37 baseline (same month last year, with fallback)
def build_w37_df(df):
    d = df.sort_values(["SEGMENT_ID", "TXN_DATE"]).copy()

    d["w37_prediction"] = d.groupby("SEGMENT_ID")["TXN_VOLUME"].shift(12)
    d["lag1"] = d.groupby("SEGMENT_ID")["TXN_VOLUME"].shift(1)
    d["rmean3"] = d.groupby("SEGMENT_ID")["TXN_VOLUME"].shift(1).rolling(3, min_periods=1).mean()

    d["w37_prediction"] = (
        d["w37_prediction"]
        .fillna(d["lag1"])
        .fillna(d["rmean3"])
        .fillna(0)
    )

    return d[["SEGMENT_ID", "TXN_DATE", "w37_prediction"]].set_index(["SEGMENT_ID", "TXN_DATE"])

w37_df = build_w37_df(df)


In [ ]:

# Plot function
def plot_predictions(part_num, prediction_df, actual_df, naive_df, w37_df):
    pred = prediction_df[prediction_df["SEGMENT_ID"] == part_num]
    act = actual_df[actual_df["SEGMENT_ID"] == part_num]

    if pred.empty and act.empty:
        return

    plt.figure(figsize=(12,5))

    if not act.empty:
        plt.plot(act["TXN_DATE"], act["actual"], label="Actual", color="black", linewidth=2)

    if not pred.empty:
        plt.plot(pred["TXN_DATE"], pred["prediction"], label="LightGBM", color="blue")

    if part_num in naive_df.index.get_level_values(0):
        n = naive_df.loc[part_num].reset_index()
        plt.plot(n["TXN_DATE"], n["naive_prediction"], "--", label="Naive", color="orange")

    if part_num in w37_df.index.get_level_values(0):
        w = w37_df.loc[part_num].reset_index()
        plt.plot(w["TXN_DATE"], w["w37_prediction"], ":", label="W-37", color="green")

    plt.title(f"SEGMENT_ID = {part_num}")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Example plots
for part in df["SEGMENT_ID"].drop_duplicates().head(5):
    plot_predictions(part, prediction_df, actual_df, naive_df, w37_df)


In [ ]:
type(m_lgb)

In [ ]:
from ibm_watson_machine_learning import APIClient

# SET DEPLOYMENT SPACE

# space_id must already exist (from earlier cell or UI)
client.set.default_space(space_id)
print("Using deployment space:", space_id)

# GET SOFTWARE SPECIFICATION

software_spec_id = client.software_specifications.get_id_by_name(
    "runtime-24.1-py3.11"
)

print("Software spec ID:", software_spec_id)

# REGISTER ML MODEL

model_metadata = {
    client.repository.ModelMetaNames.NAME: "demand_forecasting_lgbm",
    client.repository.ModelMetaNames.TYPE: "scikit-learn_1.3",
    client.repository.ModelMetaNames.SOFTWARE_SPEC_UID: software_spec_id
}

model_details = client.repository.store_model(
    model=m_lgb,          # <-- trained model
    meta_props=model_metadata
)

model_id = model_details["metadata"]["id"]
print("Model registered with ID:", model_id)

# CREATE ONLINE DEPLOYMENT

deployment_metadata = {
    client.deployments.ConfigurationMetaNames.NAME: "demand-forecasting-ml",
    client.deployments.ConfigurationMetaNames.ONLINE: {}
}

deployment_details = client.deployments.create(
    artifact_uid=model_id,
    meta_props=deployment_metadata,
    space_id=space_id
)

deployment_id = deployment_details["metadata"]["id"]
scoring_url = deployment_details["entity"]["status"]["online_url"]["url"]

print("Deployment SUCCESSFUL")
print("Deployment ID:", deployment_id)
print("Scoring URL:", scoring_url)
